# 04. Flickr30k: training the proposed generator (Dynamic Logits + contrastive learning + attention pooling)

Used for the Flickr30k results in the paper (Karpathy split, 29,000 training images, peak learning rate 3 × 10⁻⁴).

`NUM_LAYERS` is set to 3 to match the checkpoint that was used. The other constants are unchanged and have not been re-run to verify them.

Flickr30k has no COCO instance labels, so the object labels used by Dynamic Logits and by the negative captions are taken from the image's own 5 reference captions (80 COCO classes, function `build_caption_objects`). An object that is visible but named by no caption is treated as absent. Evaluation is outside this notebook: use `eval/flickr30k_eval.py`.

# CLIP-ViT + Transformer with Dynamic Logits, contrastive learning and attention pooling on Flickr30k

Port of the COCO notebook (`02_train_proposed_dynlogits_contrastive_attnpool.ipynb`) to **Flickr30k (Karpathy split)**.

Architecture and training are unchanged: Dynamic Logits on the decoder output (soft, probability-based penalty, 10-epoch warm-up), multi-positive contrastive learning (SupCon, 3 negatives: 2 object swaps + 1 NegCLIP-style word-order swap), attention pooling, label smoothing 0.1, gradient clipping 1.0, AMP.

## Differences from the COCO version

1. **Data**: `dataset_flickr30k.json` (train 29,000 / val 1,014 / test 1,000 images, 5 captions each), no restval. Paths are found automatically under `/kaggle/input`; attach the Flickr30k Karpathy split and the Flickr30k images.
2. **Object labels**: Flickr30k has no COCO instance labels, so the objects "present in the image" are taken from its 5 captions (`build_caption_objects`, 80 COCO classes). An object that is visible but named by no caption is treated as absent, so Dynamic Logits and the negatives penalise wrongly more often than on COCO. Keep this in mind when reading the results.
3. **Number of epochs**: `EPOCHS` = 30 (1,511 steps per epoch, about 3.9 times fewer than COCO).
4. **Evaluation**: not in this notebook; use `eval/flickr30k_eval.py` (logit blocking at inference + reranking, as for COCO).

Checkpoint and vocabulary names start with `flickr30k_karpathy_dynlogits_contrastive_attnpool_`.

In [ ]:
import os
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
from transformers import CLIPVisionModel, get_linear_schedule_with_warmup
import json  # json for the COCO annotations and saving the vocab

# ==========================================
# CONFIGURATION AND HYPERPARAMETERS (TRAIN FROM SCRATCH)
# ==========================================
SEED = 42

def seed_everything(seed=SEED):
    """Fix the RNG so training runs are reproducible."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # fixed IMAGE_SIZE/BATCH_SIZE -> safe to enable the cuDNN autotuner, faster

def seed_worker(worker_id):
    """Seed Python/NumPy in each DataLoader worker."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Flickr30k paths on Kaggle. None = auto-detect under /kaggle/input, so you only need to attach (Add Input) 2 datasets:
#   1) the Flickr30k Karpathy split (file dataset_flickr30k.json: train 29,000 / val 1,014 / test 1,000 images, 5 captions each),
#   2) the Flickr30k images (folder with the .jpg files, e.g. 1000092795.jpg).
# Can be filled in by hand if auto-detection fails.
KARPATHY_JSON_FILE = None
FLICKR_IMG_DIR = None
# Fallback when there is no Karpathy file: captions.txt (columns image,caption) or results.csv (image_name| comment_number| comment).
# Then images are split at random (fixed seed) 29,000 / 1,014 / 1,000, which does NOT match the Karpathy split used in papers.
FLICKR_CAPTIONS_FILE = None
# The Flickr30k Karpathy split has no restval: the training set is "train" only.
TRAIN_SPLITS = ("train",)

# Output paths - named "scratch" so they do not overwrite the checkpoints of the old fine-tune version
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_vocab.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_last_checkpoint.pth")
BEST_MODEL_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_best_model.pth")

# NEW ablation (adds the contrastive loss + a new projection head) -> train from scratch, do NOT inherit
# checkpoint of the dynamic-logits+fixloss configuration (the architecture changed, reloading would mismatch shapes).
# If Kaggle interrupts the kernel, set RESUME_TRAINING=True and point RESUME_CHECKPOINT_FILE
# to the last_checkpoint saved by THIS contrastive run (not a checkpoint from another run).
RESUME_TRAINING = False
RESUME_CHECKPOINT_FILE = CHECKPOINT_FILE
FINE_TUNE_RESET_TRAINING_STATE = False
# Continue training when the old LR schedule has finished (LR ~ 0, e.g. EPOCHS raised 10 -> 19): set a small peak LR (e.g. 2e-5)
# to build a NEW warm-up + linear-decay schedule for the remaining epochs. None = keep the old schedule.
RESUME_PEAK_LR = None

# Hallucination-aware training: Flickr30k has no COCO instances, so the image object labels come from its own 5 captions
# (see build_caption_objects). No YOLO.
USE_HALL_LOSS = False  # off: only Dynamic Logits (decoder intercept) + Contrastive are used
USE_CAPTION_OBJECT_LABELS = True
USE_YOLO_HALL_LOSS = False
LAMBDA_HALL = 0.01
HALL_LAMBDA_WARMUP_EPOCHS = 3
HALL_LOSS_MIN_STEP_RATIO = 0.3
HALL_DEBUG_BATCHES = 2

# --- Image-caption contrastive loss ---
# Positive = correct caption (teacher forcing); negative = correct caption with 1 object PRESENT
# PRESENT in the image with 1 object NOT in the image -> reuses exactly
# COCO_OBJECT_SYNONYMS/object_token_ids already exist for HalluLoss, no new data source needed.
USE_CONTRASTIVE_LOSS = True
LAMBDA_CONTRASTIVE = 0.01  # raised from 0.005 (if CHAIR does not drop enough, try lambda_con=0.01)
NUM_NEGATIVES = 3  # 2 hallucination-style (object swap) + 1 structural-style (position swap,
                   # NegCLIP style) - based on Wu et al. 2022, NegCLIP (Yuksekgonul et al. 2023).
NUM_NEGATIVES_STRUCTURAL = 1  # of NUM_NEGATIVES, how many are NegCLIP style: swap the positions of 2
                              # objects PRESENT in the caption, KEEPING the vocabulary (bag-of-words)
                              # -> forces learning of binding/word order, unlike the hallucination type
SKIP_UNSWAPPABLE_NEGATIVES = True  # True (the fix): negatives that cannot be swapped (the caption has no object token to replace)
                                   # Words are NO LONGER appended at the end (an appended caption is ungrammatical, so the model learns the shortcut "odd caption"
                                   # instead of looking at the image); such negatives are MASKED out of the InfoNCE denominator. False = old behaviour
                                   # (an object appended at the end of the caption, or a copy of the positive).
EXPAND_OBJECT_PLURALS = True  # True (the fix): add plural forms (beds, dogs, men...) present in the vocab to the token set of each
                              # object, used by Dynamic Logits, hall loss and negative generation. False = singular forms only (old behaviour).
K_POS_PER_IMAGE = 2  # multi-positive contrastive (SupCon - Khosla et al. 2020): 1 anchor (used for CE
                     # as before) + (K-1) other REAL captions of the SAME image, used only for contrastive learning.
                     # Correct captions of OTHER images in the batch automatically act as in-batch negatives,
                     # no separate generation function needed (see multi_positive_infonce_loss).
CONTRASTIVE_TAU = 0.07
CONTRASTIVE_WARMUP_EPOCHS = 3  # gate(epoch) = min(1, epoch/warmup), like HALL_LAMBDA_WARMUP_EPOCHS
CONTRASTIVE_PROJ_DIM = 256  # shared dimension the image/text embeddings are projected to before cosine similarity
POOLING_MODE = "attention"  # attention pooling: "mean" (mean pooling, the default
                            # of the old architecture) or "attention" (the AttentionPool added in
                            # this notebook). A string (not a True/False flag) so a single variable
                            # controls both project_image and project_text, convenient
                            # can switch the pooling for an ablation, e.g. re-running with POOLING_MODE="mean" on
                            # THIS notebook (the AttentionPool layer is always initialised,
                            # it is just never CALLED when POOLING_MODE="mean").
CONTRASTIVE_ATTN_POOL_HEADS = 4  # number of heads of nn.MultiheadAttention in AttentionPool,
                                 # a small extra cost (negligible compared with
                                 # the main encoder/decoder).
USE_AMP = True  # mixed precision (fp16 autocast + GradScaler) for the training loop - speed only, NOT used in the val loop so val metrics stay in full fp32 as in earlier runs

# --- Stop safely before the 12-hour Kaggle session limit ---
# At the end of each epoch (after saving last_checkpoint): stop if elapsed time + 1.1 x the last epoch time exceeds the budget
# completely, the session ends normally and the output is saved. The next session resumes with RESUME_TRAINING=True. None = no limit.
TRAIN_TIME_BUDGET_HOURS = 11.5
NOTEBOOK_START_TIME = __import__("time").time()

IMAGE_SIZE = (224, 224)
SEQ_LENGTH = 25
MIN_WORD_FREQ = 5
EMBED_DIM = 512
# --- Keep the decoder architecture IDENTICAL to the baseline/fine-tune version (FF_DIM=512, NUM_LAYERS=2) ---
# To compare the baseline with the train-from-scratch version (hallucination loss + decoder intercept + BLEU
# checkpoint selection) is fair, do NOT change the architecture here - if both change at once, a better result
# cannot be attributed to the larger decoder or to the hallucination improvements being evaluated. To try
# a larger architecture, change these 2 numbers directly and treat it as a SEPARATE experiment, NOT combined
# with the baseline comparison experiment.
FF_DIM = 512
NUM_LAYERS = 3  # 3-layer decoder, as in the checkpoint that was used
NUM_HEADS = 8
BATCH_SIZE = 96  # 32 -> 96: GPU ~1.5x faster per sample, peak memory ~2.2 GB (measured on an RTX 3050 with AMP),
                 # Plenty for Kaggle 16 GB. Note: the number of in-batch contrastive negatives triples and steps per epoch drop 3 times.

# Peak LR when training from scratch, the same value as the Flickr30k baseline for a fair comparison. The baseline's 1e-4 (30 epochs) still
# underfits: val loss drops every epoch, train ≈ val. Resuming from a checkpoint uses RESUME_PEAK_LR instead of this flag.
EPOCHS = 30  # 1,511 steps/epoch (29,000 images x 5 captions / 96); the COCO version runs 40 epochs x 5,904 steps
LEARNING_RATE = 3e-4

# --- Improvements against overfitting / hallucination when training from scratch ---
LABEL_SMOOTHING = 0.1  # reduces overconfidence -> reduces hallucination
GRAD_CLIP_NORM = 1.0   # stabilises gradients when training the decoder from scratch

# --- Dynamic intercept on the decoder output DURING TRAINING ---
# The old version used a static penalty at every timestep, which easily distorts the CE loss and can raise CHAIR.
# This dynamic version is softer:
# 1) ramp up the penalty by epoch so the first epochs learn language/grounding properly first,
# 2) only penalise strongly at timesteps where the model gives high probability to the absent-object group,
# 3) do not penalise the correct target token in the CE, avoiding conflicts when COCO instances miss an object.
TRAIN_DECODER_INTERCEPT = True
TRAIN_INTERCEPT_MODE = "dynamic"
TRAIN_INTERCEPT_MAX_PENALTY = 0.5
TRAIN_INTERCEPT_WARMUP_EPOCHS = 10
TRAIN_INTERCEPT_PROB_THRESHOLD = 0.08
TRAIN_INTERCEPT_TARGET_PROTECT = True

# --- Combined flag: do we need to build object_token_ids / COCO instance objects? ---
# 3 components share this grounding setup: HalluLoss (loss penalty), Dynamic Decoder
# Intercept (lowers logits) and the contrastive loss (builds negative captions). USE_HALL_LOSS=False does NOT
# is allowed to switch this setup off if Dynamic Intercept or Contrastive still needs it - otherwise,
# apply_decoder_object_intercept/build_negative_captions will always be no-ops even when their flags are on.
# USE_HALL_LOSS now only decides whether lambda_hall is added to the loss (see get_hall_lambda
# below, changed to current_lambda_hall = 0.0 when USE_HALL_LOSS=False).
USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS

# --- Monitor / early-stop on a real caption metric (BLEU-4) instead of only val_loss ---
# a lower val_loss does not mean better captions: use the standard COCO-caption BLEU-4 (pycocoevalcap,
# Does NOT use nltk because nltk BLEU was designed for machine translation and does not match how image
# captioning in COCO papers/benchmarks) on a subset of val images each epoch (greedy decode)
# to pick the best model by actual caption quality.
EARLY_STOP_METRIC = "loss"  # "bleu4" | "loss"
VAL_METRIC_MAX_SAMPLES = 1000

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

# ==========================================
# DATA PREPROCESSING
# ==========================================
def custom_standardization(text):
    text = text.lower()
    strip_chars = r"""!"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"""
    text = re.sub(f"[{re.escape(strip_chars)}]", "", text)
    return text.strip()

# Train: with augmentation (RandomRotation reduced from the fine-tune version: 36 -> 15 degrees, because strong rotation
# shifts the image distribution away from CLIP pretraining and does not update the caption to match the rotation
# -> label noise, especially harmful when training from scratch).
train_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=15),
    T.ColorJitter(contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

# Validation: NO augmentation
val_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

class FlickrDataset(Dataset):
    def __init__(self, image_paths, captions, vocab, transforms=None, image_objects=None, k_pos_per_image=1):
        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transforms = transforms
        self.image_objects = image_objects or {}
        self.k_pos_per_image = max(k_pos_per_image, 1)
        # Multi-positive contrastive (SupCon - Khosla et al. 2020): gather ALL real captions of the same image (e.g. 5
        # Karpathy captions) to take (k_pos_per_image - 1) extra positives for the contrastive loss - CE
        # still use exactly 1 caption per idx as before, UNCHANGED.
        self.image_to_captions = {}
        for path, cap in zip(image_paths, captions):
            self.image_to_captions.setdefault(path, []).append(cap)

    def __len__(self):
        return len(self.image_paths)

    def _tokenize(self, caption):
        tokens = [self.vocab.get(word, UNK_IDX) for word in caption.split()]
        tokens = [START_IDX] + tokens[:SEQ_LENGTH - 2] + [END_IDX]
        padded_tokens = tokens + [PAD_IDX] * (SEQ_LENGTH - len(tokens))
        return torch.tensor(padded_tokens, dtype=torch.long)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        caption = self.captions[idx]
        gt_objects_text = "|".join(sorted(self.image_objects.get(os.path.basename(img_path), set())))

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback if an image is corrupted (very rare, but possible in a large dataset)
            image = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))

        if self.transforms:
            image = self.transforms(image)

        target_tensor = self._tokenize(caption)

        num_extra = self.k_pos_per_image - 1
        if num_extra > 0:
            # Prefer a caption DIFFERENT from the anchor to avoid an identical positive; if the image has only 1
            # caption (rare), fall back to the anchor itself (no crash, only less diversity).
            candidates = [c for c in self.image_to_captions.get(img_path, []) if c != caption]
            if not candidates:
                candidates = self.image_to_captions.get(img_path, [caption])
            if len(candidates) >= num_extra:
                extra_captions = random.sample(candidates, num_extra)
            else:
                extra_captions = [random.choice(candidates) for _ in range(num_extra)]
            extra_pos_tensor = torch.stack([self._tokenize(c) for c in extra_captions], dim=0)
        else:
            extra_pos_tensor = torch.zeros((0, SEQ_LENGTH), dtype=torch.long)

        return image, target_tensor, img_path, gt_objects_text, extra_pos_tensor

# ==========================================
# FLICKR30K DATA (Karpathy split)
# ==========================================
_FLICKR_META = None   # list[(filename, split, [raw captions])]
SPLIT_SOURCE = None


def _find_first(root, predicate):
    for dirpath, _, files in os.walk(root):
        for name in files:
            if predicate(name):
                return os.path.join(dirpath, name)
    return None


def _load_flickr_meta():
    """Read the list of (filename, split, raw caption) once: prefer dataset_flickr30k.json, otherwise use the caption file + a random split."""
    global _FLICKR_META, SPLIT_SOURCE, KARPATHY_JSON_FILE, FLICKR_IMG_DIR, FLICKR_CAPTIONS_FILE
    if _FLICKR_META is not None:
        return _FLICKR_META

    if KARPATHY_JSON_FILE is None and FLICKR_CAPTIONS_FILE is None:
        KARPATHY_JSON_FILE = _find_first("/kaggle/input", lambda n: n == "dataset_flickr30k.json")
        if KARPATHY_JSON_FILE is None:
            FLICKR_CAPTIONS_FILE = (_find_first("/kaggle/input", lambda n: n == "captions.txt")
                                    or _find_first("/kaggle/input", lambda n: n == "results.csv"))
    meta = []
    if KARPATHY_JSON_FILE:
        with open(KARPATHY_JSON_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        for img in data.get("images", []):
            sents = [s["raw"] if "raw" in s else " ".join(s.get("tokens", [])) for s in img.get("sentences", [])]
            if img.get("filename") and sents:
                meta.append((img["filename"], img.get("split"), sents))
        SPLIT_SOURCE = f"Karpathy split ({KARPATHY_JSON_FILE})"
    elif FLICKR_CAPTIONS_FILE:
        by_image = {}
        with open(FLICKR_CAPTIONS_FILE, "r", encoding="utf-8") as f:
            header = f.readline()
            sep = "|" if "|" in header else ","
            for line in f:
                parts = line.rstrip("\n").split(sep, 2 if sep == "|" else 1)
                if len(parts) < 2:
                    continue
                name, cap = parts[0].strip(), parts[-1].strip()
                by_image.setdefault(name, []).append(cap)
        names = sorted(by_image)
        random.Random(SEED).shuffle(names)
        for i, name in enumerate(names):
            split = "train" if i < 29000 else "val" if i < 29000 + 1014 else "test" if i < 29000 + 1014 + 1000 else None
            if split:
                meta.append((name, split, by_image[name]))
        SPLIT_SOURCE = f"RANDOM SPLIT of {FLICKR_CAPTIONS_FILE} (not the Karpathy split)"
    else:
        raise RuntimeError("dataset_flickr30k.json or captions.txt/results.csv not found under /kaggle/input. "
                           "Attach the Flickr30k dataset or set KARPATHY_JSON_FILE / FLICKR_CAPTIONS_FILE.")

    if FLICKR_IMG_DIR is None:
        probe = meta[0][0]
        hit = _find_first("/kaggle/input", lambda n: n == probe)
        if hit is None:
            raise RuntimeError(f"Image {probe} not found under /kaggle/input. Attach the Flickr30k image dataset or set FLICKR_IMG_DIR.")
        FLICKR_IMG_DIR = os.path.dirname(hit)
    _FLICKR_META = meta
    return meta


_DIR_FILES = {}


def _exists_fast(path):
    """Replacement for os.path.exists on image paths: list the folder once and look up in a set (/kaggle/input is very slow when
    stat-ing each file). A folder that does not exist counts as empty."""
    d, name = os.path.split(path)
    if d not in _DIR_FILES:
        try:
            _DIR_FILES[d] = set(os.listdir(d))
        except (FileNotFoundError, NotADirectoryError):
            _DIR_FILES[d] = set()
    return name in _DIR_FILES[d]


def load_flickr_data(split="train"):
    """Return (image_paths, captions) as image-caption pairs; split is one name or a tuple of names."""
    splits = {split} if isinstance(split, str) else set(split)
    image_paths, captions = [], []
    missing = 0
    for filename, sp, sents in _load_flickr_meta():
        if sp not in splits:
            continue
        path = os.path.join(FLICKR_IMG_DIR, filename)
        if not _exists_fast(path):
            missing += 1
            continue
        for raw in sents:
            caption = custom_standardization(raw)
            if caption:
                image_paths.append(path)
                captions.append(caption)
    print(f"-> Flickr30k {sorted(splits)}: {len(set(image_paths))} images, {len(captions)} image-caption pairs.")
    if missing:
        print(f"-> Skipped {missing} images whose files were not found.")
    return image_paths, captions


def build_or_load_vocab(captions, min_freq=5, vocab_file=VOCAB_FILE):
    """Build a new vocabulary, or load it from file if it already exists"""
    if os.path.exists(vocab_file):
        print(f"-> Found the vocab file. Loading the vocabulary from {vocab_file}...")
        with open(vocab_file, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"-> Loaded vocabulary size: {len(vocab)}")
        return vocab

    print(f"-> File {vocab_file} not found. Building a new vocabulary...")
    all_words = []
    for cap in captions:
        all_words.extend(cap.split())

    word_counts = Counter(all_words)
    valid_words = [word for word, count in word_counts.items() if count >= min_freq]

    vocab = {"<pad>": PAD_IDX, "<start>": START_IDX, "<end>": END_IDX, "<unk>": UNK_IDX}
    for idx, word in enumerate(valid_words):
        vocab[word] = idx + 4

    # Save the vocab for reuse
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)

    print(f"-> Vocabulary size (words appearing >= {min_freq} times): {len(vocab)}")
    print(f"-> Saved the vocabulary to: {vocab_file}")
    return vocab

In [ ]:
# ==========================================
# MODEL ARCHITECTURE
# ==========================================

# ==========================================
# HALLUCINATION-AWARE LOSS BASED ON COCO INSTANCES (GROUND TRUTH)
# ==========================================
COCO_OBJECT_SYNONYMS = {
    "person": ["person", "people", "man", "woman", "boy", "girl", "child", "children"],
    "bicycle": ["bicycle", "bike"], "car": ["car", "auto", "automobile"],
    "motorcycle": ["motorcycle", "motorbike"], "airplane": ["airplane", "plane", "aircraft"],
    "bus": ["bus"], "train": ["train"], "truck": ["truck"], "boat": ["boat"],
    "traffic light": ["traffic", "light"], "fire hydrant": ["hydrant"], "stop sign": ["stop", "sign"],
    "parking meter": ["meter"], "bench": ["bench"], "bird": ["bird"], "cat": ["cat"], "dog": ["dog"],
    "horse": ["horse"], "sheep": ["sheep"], "cow": ["cow"], "elephant": ["elephant"], "bear": ["bear"],
    "zebra": ["zebra"], "giraffe": ["giraffe"], "backpack": ["backpack"], "umbrella": ["umbrella"],
    "handbag": ["handbag", "bag", "purse"], "tie": ["tie"], "suitcase": ["suitcase", "luggage"],
    "frisbee": ["frisbee"], "skis": ["skis"], "snowboard": ["snowboard"], "sports ball": ["ball"],
    "kite": ["kite"], "baseball bat": ["bat"], "baseball glove": ["glove"], "skateboard": ["skateboard"],
    "surfboard": ["surfboard"], "tennis racket": ["racket"], "bottle": ["bottle"], "wine glass": ["glass"],
    "cup": ["cup"], "fork": ["fork"], "knife": ["knife"], "spoon": ["spoon"], "bowl": ["bowl"],
    "banana": ["banana"], "apple": ["apple"], "sandwich": ["sandwich"], "orange": ["orange"],
    "broccoli": ["broccoli"], "carrot": ["carrot"], "hot dog": ["hotdog"], "pizza": ["pizza"],
    "donut": ["donut", "doughnut"], "cake": ["cake"], "chair": ["chair"], "couch": ["couch", "sofa"],
    "potted plant": ["plant"], "bed": ["bed"], "dining table": ["table"], "toilet": ["toilet"],
    "tv": ["tv", "television"], "laptop": ["laptop"], "mouse": ["mouse"], "remote": ["remote"],
    "keyboard": ["keyboard"], "cell phone": ["phone", "cellphone"], "microwave": ["microwave"],
    "oven": ["oven"], "toaster": ["toaster"], "sink": ["sink"], "refrigerator": ["refrigerator", "fridge"],
    "book": ["book"], "clock": ["clock"], "vase": ["vase"], "scissors": ["scissors"],
    "teddy bear": ["teddy"], "hair drier": ["drier", "dryer"], "toothbrush": ["toothbrush"],
}

# Plural forms NOT added to the object table because their meaning differs from the COCO object:
# "glasses" in captions usually means eyeglasses, not a wine glass.
_PLURAL_EXCLUDE = {"glasses"}
# Misspelled forms present in the vocab: still used to DETECT/penalise tokens but NEVER picked as replacements when building negatives
# (captions such as "next to a mans" are ungrammatical, so the model spots an "odd caption" without looking at the image).
_REPLACEMENT_TYPOS = {"mans", "womans", "childs", "childrens", "peoples", "persons", "knifes", "mouses", "buss", "sheeps"}
_BASE_PLURAL_WORDS = {"people", "children"}  # plural forms already present in COCO_OBJECT_SYNONYMS
OBJECT_PLURAL_IDS = set()       # ids of plural tokens (filled in build_object_token_ids when EXPAND_OBJECT_PLURALS)
REPLACEMENT_EXCLUDE_IDS = set()  # ids of misspelled tokens, never used as replacements
_PLURAL_IRREGULAR = {"man": ["men"], "woman": ["women"], "child": ["children"], "person": ["people", "persons"],
                      "mouse": ["mice"], "knife": ["knives"]}


def _plural_forms(word):
    """Possible plural forms of a single word (multi-word phrases are skipped). Only forms PRESENT in the vocab are used."""
    if " " in word:
        return []
    forms = list(_PLURAL_IRREGULAR.get(word, []))
    forms += [word + "s", word + "es"]
    if word.endswith("y") and len(word) > 1 and word[-2] not in "aeiou":
        forms.append(word[:-1] + "ies")
    if word.endswith("f"):
        forms.append(word[:-1] + "ves")
    if word.endswith("fe"):
        forms.append(word[:-2] + "ves")
    return forms


# category -> list of words (singular + plural if EXPAND_OBJECT_PLURALS), filled in build_object_token_ids.
OBJECT_WORDS = {}


def build_object_token_ids(vocab):
    global OBJECT_WORDS, OBJECT_PLURAL_IDS, REPLACEMENT_EXCLUDE_IDS
    added_plurals = set()
    base_words = {w for words in COCO_OBJECT_SYNONYMS.values() for w in words}
    taken = set(base_words)  # do not let a plural form "steal" the base word of another category or of a category taken earlier
    object_token_ids = {}
    OBJECT_WORDS = {}
    for obj_name, synonyms in COCO_OBJECT_SYNONYMS.items():
        words = list(synonyms)
        if EXPAND_OBJECT_PLURALS:
            for w in synonyms:
                for p in _plural_forms(w):
                    if p in vocab and p not in taken and p not in _PLURAL_EXCLUDE:
                        words.append(p)
                        taken.add(p)
                        added_plurals.add(p)
        OBJECT_WORDS[obj_name] = words
        ids = sorted({vocab[word] for word in words if word in vocab})
        if ids:
            object_token_ids[obj_name] = ids
    if EXPAND_OBJECT_PLURALS:
        OBJECT_PLURAL_IDS = {vocab[w] for w in (added_plurals | _BASE_PLURAL_WORDS) if w in vocab}
        REPLACEMENT_EXCLUDE_IDS = {vocab[w] for w in _REPLACEMENT_TYPOS if w in vocab}
    else:
        OBJECT_PLURAL_IDS, REPLACEMENT_EXCLUDE_IDS = set(), set()
    print(f"-> Hall loss tracks {len(object_token_ids)} objects that have tokens in the vocabulary.")
    return object_token_ids


def build_caption_objects(image_paths, captions):
    """Flickr30k has no COCO instances, so there are no human-drawn object labels. They are replaced by labels taken from the image's own 5 captions:
    an object (among the 80 COCO classes) counts as PRESENT in the image if at least 1 of its captions names it (via OBJECT_WORDS, plurals included).
    Consequence to keep in mind: an object that is visible but named by no caption is treated as absent, so Dynamic Logits and the negatives will penalise
    wrongly more often than with COCO instance labels. Keys are file names, as in the COCO version."""
    word_to_category = {}
    for category, words in OBJECT_WORDS.items():
        for w in words:
            word_to_category.setdefault(w, category)
    image_objects = {}
    for path, cap in zip(image_paths, captions):
        found = image_objects.setdefault(os.path.basename(path), set())
        for w in cap.split():
            category = word_to_category.get(w)
            if category is not None:
                found.add(category)
    print(f"-> Took object labels from captions for {len(image_objects)} images "
          f"(average {sum(len(v) for v in image_objects.values()) / max(len(image_objects), 1):.2f} objects/image).")
    return image_objects

HALL_DEBUG_STATE = {"calls": 0}

def hallucination_object_loss(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    token_mask=None,
    min_step_index=0,
):
    """Object-level unlikelihood loss, softened to work together with dynamic logits.

    - Use the mean over the absent-token group instead of the sum so that images with many absent objects
      is not penalised heavily just because there are more tokens to consider.
    - Only penalise from min_step_index onwards to avoid noise at the start of the caption, when the decoder still has
      little language context.
    """
    if not object_token_ids:
        return outputs.new_tensor(0.0)

    probs = outputs.softmax(dim=-1)
    seq_len = outputs.size(1)
    if min_step_index > 0:
        step_positions = torch.arange(seq_len, device=outputs.device)
        step_position_mask = (step_positions >= min_step_index).float()
    else:
        step_position_mask = None

    losses = []
    num_with_gt_objects = 0
    num_with_present_objects = 0
    num_with_absent_tokens = 0
    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        gt_objects = set(gt_text.split("|")) if gt_text else set()
        if gt_objects:
            present_objects = gt_objects
            num_with_gt_objects += 1
            num_with_present_objects += 1
        else:
            continue

        absent_token_ids = []
        for obj_name, token_ids in object_token_ids.items():
            if obj_name not in present_objects:
                absent_token_ids.extend(token_ids)

        if not absent_token_ids:
            continue
        num_with_absent_tokens += 1

        absent_ids = torch.tensor(sorted(set(absent_token_ids)), device=outputs.device, dtype=torch.long)
        absent_probs = probs[batch_index].index_select(dim=-1, index=absent_ids).clamp(max=1.0 - 1e-6)
        step_penalty = -torch.log1p(-absent_probs).mean(dim=-1)

        if token_mask is not None:
            valid_steps = token_mask[batch_index].float()
        else:
            valid_steps = outputs.new_ones(step_penalty.shape)

        if step_position_mask is not None:
            valid_steps = valid_steps * step_position_mask

        valid_steps_sum = valid_steps.sum()
        if valid_steps_sum.item() <= 0:
            continue

        losses.append((step_penalty * valid_steps).sum() / valid_steps_sum.clamp_min(1.0))

    if not losses:
        if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
            print(
                f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
                f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
                f"penalized_images={num_with_absent_tokens} | hall_ul=0.00000000"
            )
            HALL_DEBUG_STATE["calls"] += 1
        return outputs.new_tensor(0.0)

    if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
        debug_value = torch.stack(losses).mean().detach().item()
        print(
            f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
            f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
            f"penalized_images={num_with_absent_tokens} | hall_ul={debug_value:.8f}"
        )
        HALL_DEBUG_STATE["calls"] += 1
    return torch.stack(losses).mean()


def get_hall_lambda(epoch, max_lambda=LAMBDA_HALL, warmup_epochs=HALL_LAMBDA_WARMUP_EPOCHS):
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


# ==========================================
# IMAGE-CAPTION CONTRASTIVE LOSS
# ==========================================
def build_object_id_to_name(object_token_ids):
    """Invert object_token_ids (obj_name -> [token_id,...]), already built for HalluLoss, into
    token_id -> obj_name, used to find object tokens in a caption when building negatives."""
    id_to_name = {}
    for obj_name, token_ids in object_token_ids.items():
        for token_id in token_ids:
            id_to_name[token_id] = obj_name
    return id_to_name


def build_object_frequency_weights(captions, object_token_ids):
    """Count the REAL frequency of each category (over all its synonyms) in the ground-truth captions
    ground truth (instead of listing all 80 categories uniformly as before). Based on Rohrbach et al. 2018 -
    real hallucination correlates with how often an object appears in the data (e.g. "dining table" is
    hallucinated most because it appears very often in captions) - see
    Used as category sampling weights instead of
    uniformly over the 80 categories, which makes the negatives "harder" and more realistic.

    Returns a dict category -> count (>= 1, add-one Laplace smoothing so rare categories still have
    a chance of being chosen, never weight = 0).
    """
    word_to_category = {}
    for category in object_token_ids.keys():
        for word in (OBJECT_WORDS.get(category) or COCO_OBJECT_SYNONYMS.get(category, [])):
            word_to_category[word] = category

    weights = {name: 1 for name in object_token_ids.keys()}
    for caption in captions:
        seen_categories = set()
        for word in caption.split():
            category = word_to_category.get(word)
            if category is not None:
                seen_categories.add(category)
        for category in seen_categories:
            weights[category] += 1
    return weights


def _weighted_sample_without_replacement(names, weights, k):
    """Draw k elements WITHOUT replacement from names, weighted by weights (algorithm
    Efraimidis-Spirakis: key = U^(1/w), take the k largest keys - equivalent to weighted sampling
    without replacement). weights=None or empty -> uniform (plain random.sample)."""
    if not names:
        return []
    if not weights:
        return random.sample(names, min(k, len(names)))
    keyed = [
        (random.random() ** (1.0 / max(weights.get(name, 1), 1e-6)), name)
        for name in names
    ]
    keyed.sort(key=lambda pair: pair[0], reverse=True)
    return [name for _, name in keyed[:k]]


def _pick_replacement_token(token_ids, original_token):
    """Pick a replacement token with the SAME NUMBER (singular/plural) as the replaced token so the caption keeps number agreement (e.g. not "a dogs"),
    and never picks a misspelled form. Only active with EXPAND_OBJECT_PLURALS; when off it is identical to the old version (random.choice)."""
    if not EXPAND_OBJECT_PLURALS or not OBJECT_PLURAL_IDS:
        return random.choice(token_ids)
    pool = [t for t in token_ids if t not in REPLACEMENT_EXCLUDE_IDS]
    if original_token is not None:
        same_number = [t for t in pool if (t in OBJECT_PLURAL_IDS) == (original_token in OBJECT_PLURAL_IDS)]
        pool = same_number or pool
    return random.choice(pool or token_ids)


def build_negative_captions(
    captions, gt_objects_batch, object_token_ids, id_to_obj_name,
    num_negatives=1, num_structural=0, object_freq_weights=None, return_valid=False,
):
    """Build num_negatives negative captions per positive for the contrastive loss, of 2 types (sections 5.1/5.2
    related work, Rohrbach et al. 2018):

    Type 1 - "hallucination" (the first num_negatives - num_structural negatives): replace 1 object PRESENT
    really in the image with 1 object NOT in the image. The replacement category is chosen by
    object_freq_weights (REAL frequencies in the ground-truth captions, see build_object_frequency_weights)
    instead of sampling uniformly over the 80 categories - more realistic/harder negatives that follow the
    skews real hallucination (Rohrbach et al. 2018). The replacement word is also drawn at random
    across ALL synonyms of the category (not just the first synonym).

    If the caption contains no known object token (swap_pos):
      - SKIP_UNSWAPPABLE_NEGATIVES = True (the fix, default): NO negative is created in this case. The caption is
        kept in the tensor, but `negative_valid` = False so multi_positive_infonce_loss drops it
        from the denominator. Reason: appending a noun to a complete sentence makes it ungrammatical, so the model
        only has to notice the "odd" caption to tell it apart, without looking at the image (a shortcut).
      - SKIP_UNSWAPPABLE_NEGATIVES = False (old behaviour): fallback is a REAL INSERTION (the sequence is
        shifted one slot right, no overwrite) an absent object before the first <end>/<pad>, ONLY when at least
        at least 1 spare <pad> slot; otherwise keep the caption as is (negative == positive).
    (Even earlier, the code OVERWROTE the last token whatever its word type -> broke the grammar.)

    Type 2 - "structural" (the last num_structural negatives, NegCLIP style - Yuksekgonul et al. 2023):
    swap the POSITIONS of 2 different objects BOTH PRESENT in the caption, KEEPING all words
    vocabulary (same bag of words). If there are fewer than 2 objects to swap, keep the caption
    in that negative (like the fallback above).

    Returns a tensor (B, num_negatives, L); if return_valid=True returns (tensor, negative_valid) with
    negative_valid (B, num_negatives) bool = the negative DIFFERS from the positive and is usable in InfoNCE.
    """
    batch_size = captions.size(0)
    negative_captions = captions.unsqueeze(1).repeat(1, num_negatives, 1).clone()
    all_object_names = list(object_token_ids.keys())
    num_hallucination = max(num_negatives - num_structural, 0)

    for batch_index in range(batch_size):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        seq = captions[batch_index].tolist()

        # --- Type 1: hallucination (swap the object, favouring categories frequent in the ground-truth captions) ---
        if num_hallucination > 0:
            absent_names = [name for name in all_object_names if name not in present_objects]
            if absent_names:
                if len(absent_names) >= num_hallucination:
                    chosen_names = _weighted_sample_without_replacement(
                        absent_names, object_freq_weights, num_hallucination
                    )
                else:
                    pool_weights = (
                        [object_freq_weights.get(name, 1) for name in absent_names]
                        if object_freq_weights else None
                    )
                    chosen_names = random.choices(absent_names, weights=pool_weights, k=num_hallucination)

                swap_pos = next(
                    (pos for pos, tok in enumerate(seq) if id_to_obj_name.get(tok) in present_objects),
                    None,
                )
                can_insert = False
                insert_pos = None
                if swap_pos is None:
                    # Fallback: no object token to replace -> try INSERTING (without overwriting)
                    # absent object before the first <end>/<pad>. Only safe when the last token
                    # last token of the sequence is <pad> (there is a spare slot to "sacrifice" when shifting right by one),
                    # instead of overwriting an arbitrary word (possibly not a noun) as before.
                    insert_pos = next((pos for pos, tok in enumerate(seq) if tok in (END_IDX, PAD_IDX)), None)
                    can_insert = (not SKIP_UNSWAPPABLE_NEGATIVES) and insert_pos is not None \
                        and insert_pos > 1 and seq[-1] == PAD_IDX

                for neg_index, replacement_name in enumerate(chosen_names):
                    replacement_id = _pick_replacement_token(
                        object_token_ids[replacement_name], seq[swap_pos] if swap_pos is not None else None
                    )
                    if swap_pos is not None:
                        negative_captions[batch_index, neg_index, swap_pos] = replacement_id
                    elif can_insert:
                        # Real insertion: shift seq[insert_pos:-1] one slot to the right, then write the new token
                        # at insert_pos, so no original word is lost or overwritten (only one spare slot is "sacrificed"
                        # extra <pad> at the end of the sequence).
                        negative_captions[batch_index, neg_index, insert_pos + 1:] = \
                            negative_captions[batch_index, neg_index, insert_pos:-1].clone()
                        negative_captions[batch_index, neg_index, insert_pos] = replacement_id
                    # If no swap is possible and insertion is unsafe, keep the caption as is (negative ==
                    # positive, loss ~ ln(2) constant, grammar not broken).

        # --- Type 2: structural (swap positions, same vocabulary - NegCLIP style) ---
        if num_structural > 0:
            object_positions = [pos for pos, tok in enumerate(seq) if id_to_obj_name.get(tok) in present_objects]
            distinct_pairs = [
                (a, b) for i, a in enumerate(object_positions) for b in object_positions[i + 1:]
                if seq[a] != seq[b]
            ]
            chosen_pairs = random.sample(distinct_pairs, min(num_structural, len(distinct_pairs))) if distinct_pairs else []

            for k, (pos_a, pos_b) in enumerate(chosen_pairs):
                slot = num_hallucination + k
                negative_captions[batch_index, slot, pos_a] = seq[pos_b]
                negative_captions[batch_index, slot, pos_b] = seq[pos_a]
            # Remaining structural slots (if there are not enough distinct object pairs to swap) keep
            # unchanged (negative == positive, loss ~ ln(2) constant, no crash).

    if return_valid:
        if SKIP_UNSWAPPABLE_NEGATIVES:
            negative_valid = (negative_captions != captions.unsqueeze(1)).any(dim=-1)
        else:
            negative_valid = torch.ones(batch_size, num_negatives, dtype=torch.bool, device=captions.device)
        return negative_captions, negative_valid
    return negative_captions


def get_contrastive_lambda(epoch, max_lambda=LAMBDA_CONTRASTIVE, warmup_epochs=CONTRASTIVE_WARMUP_EPOCHS):
    """gate(epoch) * lambda_con, same warm-up formula as get_hall_lambda."""
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


def multi_positive_infonce_loss(image_embed, text_embed, num_positives, tau=CONTRASTIVE_TAU, negative_valid=None):
    """SupCon-style multi-positive InfoNCE (Khosla et al. 2020, arXiv:2004.11362) + in-batch
    negatives (multi-positive contrastive with hard negatives). Generalises the old 1-positive InfoNCE to
    K positives per image: still gives exactly the same result when num_positives=1 and batch_size=1 (no
    in-batch negative).

    text_embed: (B, K+N, D) - the K REAL positive embeddings (first, the CE anchor + extra positives
    same image), then the N synthetic negative embeddings (last, from build_negative_captions) - laid out
    in exactly this order at the call site.

    A(i) = the positives of ALL images (including other images in the batch - in-batch negatives, free,
    no separate generation function needed) + the synthetic negatives OF image i ITSELF. Synthetic negatives of
    OTHER images are EXCLUDED from A(i) (as the formula requires, because negatives generated for another image are not
    surely a valid negative for this image).

    negative_valid (B, N) bool, optional: False = that negative (of image b) is unusable (could not be swapped
    so it was dropped) -> removed from the denominator. Positives are always valid.
    """
    batch_size, total_slots, dim = text_embed.shape
    flat_text = text_embed.reshape(batch_size * total_slots, dim)
    sim = (image_embed @ flat_text.t()) / tau
    sim = sim.view(batch_size, batch_size, total_slots)

    slot_is_positive = torch.arange(total_slots, device=sim.device) < num_positives
    is_own_image = torch.eye(batch_size, dtype=torch.bool, device=sim.device)

    pos_mask = slot_is_positive.view(1, 1, total_slots) & is_own_image.unsqueeze(-1)
    denom_mask = slot_is_positive.view(1, 1, total_slots) | is_own_image.unsqueeze(-1)
    if negative_valid is not None:
        # sim[i, b, s]: image i against slot s of image b -> slot_valid[b, s] is selected along dim 2
        slot_valid = torch.ones(batch_size, total_slots, dtype=torch.bool, device=sim.device)
        slot_valid[:, num_positives:] = negative_valid.to(device=sim.device, dtype=torch.bool)
        denom_mask = denom_mask & slot_valid.unsqueeze(0)

    sim_flat = sim.view(batch_size, batch_size * total_slots)
    pos_mask_flat = pos_mask.reshape(batch_size, batch_size * total_slots)
    denom_mask_flat = denom_mask.reshape(batch_size, batch_size * total_slots)

    denom = torch.logsumexp(sim_flat.masked_fill(~denom_mask_flat, float("-inf")), dim=1)
    pos_sim_sum = (sim_flat * pos_mask_flat).sum(dim=1)
    pos_count = pos_mask_flat.sum(dim=1).clamp_min(1)
    mean_pos_sim = pos_sim_sum / pos_count

    loss_per_anchor = denom - mean_pos_sim
    return loss_per_anchor.mean()


def _intercept_epoch_scale(epoch_index):
    if TRAIN_INTERCEPT_WARMUP_EPOCHS <= 0:
        return 1.0
    return min(1.0, float(epoch_index + 1) / float(TRAIN_INTERCEPT_WARMUP_EPOCHS))


def apply_decoder_object_intercept(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    targets=None,
    epoch_index=0,
):
    """Dynamic logit adjustment for tokens of absent objects.

    Unlike the old static version, the penalty is no longer a constant applied at every timestep. For each image:
    - find the objects that are not in the COCO instances,
    - measure the total probability the model currently assigns to tokens of absent objects at each timestep,
    - only lower logits strongly when the probability of this group exceeds a threshold,
    - protect the correct target token so CE is not penalised backwards when an annotation misses an object.
    """
    if (
        not TRAIN_DECODER_INTERCEPT
        or TRAIN_INTERCEPT_MODE == "off"
        or not object_token_ids
    ):
        return outputs

    epoch_scale = _intercept_epoch_scale(epoch_index)
    if epoch_scale <= 0:
        return outputs

    # fixed fp32 (does not inherit the dtype of outputs) - under AMP, outputs is Half but
    # step_penalty computed from probs=outputs.softmax(...) is Float32 (autocast keeps softmax
    # kept in fp32 for numerical stability) -> assigning Half=Float used to crash with "Index put requires the
    # source and destination dtypes match". Cast to fp32 throughout, and only cast back to the dtype of
    # outputs only once, at return.
    full_bias = torch.zeros_like(outputs, dtype=torch.float32)
    probs = outputs.detach().softmax(dim=-1)
    max_penalty = TRAIN_INTERCEPT_MAX_PENALTY * epoch_scale

    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        if not present_objects:
            continue

        absent_ids = [
            token_id
            for obj_name, token_ids in object_token_ids.items()
            if obj_name not in present_objects
            for token_id in token_ids
        ]
        if not absent_ids:
            continue

        absent_ids_t = torch.tensor(sorted(set(absent_ids)), device=outputs.device, dtype=torch.long)

        if TRAIN_INTERCEPT_MODE == "static":
            step_penalty = torch.full((outputs.size(1),), max_penalty, device=outputs.device, dtype=torch.float32)
        else:
            absent_mass = probs[batch_index].index_select(dim=-1, index=absent_ids_t).sum(dim=-1)
            normalized = (
                (absent_mass - TRAIN_INTERCEPT_PROB_THRESHOLD)
                / max(1e-6, 1.0 - TRAIN_INTERCEPT_PROB_THRESHOLD)
            ).clamp(min=0.0, max=1.0)
            step_penalty = max_penalty * normalized

        full_bias[batch_index, :, absent_ids_t] = -step_penalty.unsqueeze(-1)

        if TRAIN_INTERCEPT_TARGET_PROTECT and targets is not None:
            valid = targets[batch_index] != PAD_IDX
            timesteps = torch.arange(outputs.size(1), device=outputs.device)[valid]
            target_ids = targets[batch_index, valid]
            full_bias[batch_index, timesteps, target_ids] = 0.0

    return outputs + full_bias.to(outputs.dtype)

class CLIPEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPEncoder, self).__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch16",
            use_safetensors=True,
        )

        for param in self.clip_vision.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(768, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        outputs = self.clip_vision(pixel_values=images)
        features = outputs.last_hidden_state
        embeddings = self.fc(features)
        return self.layer_norm(embeddings)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class AttentionPool(nn.Module):
    """Attention pooling: replaces `hidden_states.mean(dim=1)`. One learned query vector,
    attends over the whole token/patch sequence, then takes a weighted sum - keeping fine-grained information
    instead of weighting every position equally (old mean pooling). One shared layer for both image
    (CLIP-ViT patches, no key_padding_mask needed since every patch is valid) and the caption (decoder
    hidden states, needs key_padding_mask to ignore the <pad> positions, like tgt_key_padding_mask).

    Pooling learned per side, used for the contrastive projection heads."""
    def __init__(self, dim, num_heads=CONTRASTIVE_ATTN_POOL_HEADS):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, batch_first=True)

    def forward(self, sequence, key_padding_mask=None):
        batch_size = sequence.size(0)
        q = self.query.expand(batch_size, -1, -1)
        pooled, _ = self.attn(q, sequence, sequence, key_padding_mask=key_padding_mask)
        return pooled.squeeze(1)  # (B, dim)


class ImageCaptioningTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, ff_dim, num_heads, num_layers, contrastive_proj_dim=CONTRASTIVE_PROJ_DIM):
        super(ImageCaptioningTransformer, self).__init__()
        self.encoder = CLIPEncoder(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)

        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

        # --- Projection head for the contrastive loss ---
        # NEW parameters, absent from old checkpoints -> load_checkpoint() must use strict=False.
        self.image_proj = nn.Linear(embed_dim, contrastive_proj_dim)
        self.text_proj = nn.Linear(embed_dim, contrastive_proj_dim)
        # Attention pooling: AttentionPool is always initialised (so checkpoints load regardless
        # of POOLING_MODE); it is only NOT called in project_image/project_text when
        # POOLING_MODE == "mean" (see the 2 methods below).
        self.image_pool = AttentionPool(embed_dim)
        self.text_pool = AttentionPool(embed_dim)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def encode_image(self, images):
        """Separate image-encoding step so a single `memory` can be shared by the whole forward pass for
        positive AND negative captions in the contrastive loss (avoids running CLIP-ViT forward twice)."""
        return self.encoder(images)

    def decode(self, memory, captions, return_hidden=False):
        """Separate decode step so the contrastive loss can read the decoder hidden states (before
        fc_out) as the text representation, not just the logits."""
        tgt_embed = self.embedding(captions)
        tgt_embed = self.pos_encoder(tgt_embed)
        tgt_mask = self.generate_square_subsequent_mask(captions.size(1)).to(captions.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        hidden = self.transformer_decoder(tgt=tgt_embed, memory=memory,
                                          tgt_mask=tgt_mask,
                                          tgt_key_padding_mask=tgt_key_padding_mask)
        logits = self.fc_out(hidden)
        if return_hidden:
            return logits, hidden
        return logits

    def forward(self, images, captions, neg_captions=None, return_hidden=False):
        """neg_captions=None: old behaviour (non-contrastive branch / generate).
        neg_captions given, shape (B, N, L): encode the image once, then decode BOTH the positives and
        ALL N negatives in the SAME forward() call (memory is repeated N times along the batch
        before decoding), so nn.DataParallel scatter/gather works across multiple GPUs (previously the code
        calling model_.encode_image()/model_.decode() directly, bypassing DataParallel.forward entirely
        safe -> ran on a single GPU only). N=1 gives exactly the same result as before multi-negative support."""
        memory = self.encode_image(images)
        outputs, hidden = self.decode(memory, captions, return_hidden=True)
        if neg_captions is None:
            if return_hidden:
                return outputs, hidden
            return outputs

        batch_size, num_negatives, neg_len = neg_captions.shape
        neg_flat = neg_captions.reshape(batch_size * num_negatives, neg_len)
        memory_flat = memory.unsqueeze(1).expand(-1, num_negatives, -1, -1).reshape(
            batch_size * num_negatives, memory.size(1), memory.size(2)
        )
        _, neg_hidden_flat = self.decode(memory_flat, neg_flat, return_hidden=True)
        neg_text_embed = self.project_text(neg_hidden_flat, neg_flat == PAD_IDX).reshape(
            batch_size, num_negatives, -1
        )
        image_embed = self.project_image(memory)
        pos_text_embed = self.project_text(hidden, captions == PAD_IDX)
        return outputs, hidden, image_embed, pos_text_embed, neg_text_embed

    def project_image(self, memory):
        """v = normalize(W_v * pool(image_features)). pool = AttentionPool when
        POOLING_MODE="attention" (new), or mean(dim=1) when POOLING_MODE="mean" (mean pooling)."""
        if POOLING_MODE == "attention":
            pooled = self.image_pool(memory)
        else:
            pooled = memory.mean(dim=1)
        return nn.functional.normalize(self.image_proj(pooled), dim=-1)

    def project_text(self, hidden, padding_mask):
        """t = normalize(W_t * pool(decoder_hidden_states)). pool = AttentionPool
        when POOLING_MODE="attention" (key_padding_mask=padding_mask, meaning True=ignore
        the <pad> positions, like tgt_key_padding_mask), or mean pooling over the NON-pad positions
        when POOLING_MODE="mean" (mean pooling)."""
        if POOLING_MODE == "attention":
            pooled = self.text_pool(hidden, key_padding_mask=padding_mask)
        else:
            valid = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        return nn.functional.normalize(self.text_proj(pooled), dim=-1)

In [ ]:
def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _prepare_save_path(filepath):
    """Make sure the checkpoint is saved to a writable folder, not /kaggle/input."""
    if os.path.isdir(filepath):
        raise RuntimeError(
            f"'{filepath}' is a directory, not a checkpoint file. "
            "Pass CHECKPOINT_FILE or BEST_MODEL_FILE, e.g. /kaggle/working/last_checkpoint.pth."
        )
    save_dir = os.path.dirname(filepath) or "."
    os.makedirs(save_dir, exist_ok=True)
    if not os.access(save_dir, os.W_OK):
        raise RuntimeError(
            f"Cannot write the checkpoint to '{save_dir}'. "
            "On Kaggle, save to /kaggle/working and only load checkpoints from /kaggle/input."
        )
    return filepath

def _resolve_checkpoint_path(filepath):
    """Allow RESUME_CHECKPOINT_FILE to be a .pth file or a folder containing a checkpoint."""
    if os.path.isfile(filepath):
        return filepath

    if os.path.isdir(filepath):
        preferred_names = ["last_checkpoint.pth", "best_model.pth"]
        for name in preferred_names:
            candidate = os.path.join(filepath, name)
            if os.path.isfile(candidate):
                return candidate

        checkpoint_files = []
        for root, _, files in os.walk(filepath):
            for name in files:
                if name.endswith((".pth", ".pt")):
                    checkpoint_files.append(os.path.join(root, name))

        if checkpoint_files:
            checkpoint_files.sort(key=os.path.getmtime, reverse=True)
            return checkpoint_files[0]

        raise RuntimeError(f"'{filepath}' is a directory but contains no .pth/.pt file.")

    return filepath

class EarlyStoppingCallback:
    """Compared with the fine-tune version: supports mode="max" to early-stop / pick the best model by BLEU-4
    (higher score is better) instead of only mode="min" on val_loss.

    default patience 5 (instead of 3): BLEU-4 from greedy decoding on a subset each epoch is much noisier
    than val_loss (a discrete process, not smooth/monotonic like the loss) - a low patience easily
    stops early in the middle of a temporary fluctuation and misses further improvement."""
    def __init__(self, patience=5, min_delta=0.0, save_path=BEST_MODEL_FILE, mode="min"):
        assert mode in ("min", "max")
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.mode = mode
        self.counter = 0
        self.best_score = float('inf') if mode == "min" else float('-inf')
        self.early_stop = False

    def _improved(self, score):
        if self.mode == "min":
            return score < self.best_score - self.min_delta
        return score > self.best_score + self.min_delta

    def __call__(self, score, model, optimizer, epoch, scheduler=None, scaler=None):
        if self._improved(score):
            self.best_score = score
            self.counter = 0
            self.save_best_model(model, optimizer, epoch, score, scheduler, scaler)
        else:
            self.counter += 1
            print(f" EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_best_model(self, model, optimizer, epoch, score, scheduler=None, scaler=None):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': _unwrap_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            # Without these two states, resuming from the best model would warm the LR up again from the start.
            'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
            'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
            'seed': SEED,
            'monitor_metric': EARLY_STOP_METRIC,
            'best_score': self.best_score,
        }
        save_path = _prepare_save_path(self.save_path)
        torch.save(checkpoint, save_path)
        print(f" => {EARLY_STOP_METRIC} improved ({score:.4f}). Saved BEST MODEL to: {save_path}")

# ==========================================
# CHECKPOINT MANAGEMENT AND TRAINING
# ==========================================
def save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, filepath=CHECKPOINT_FILE, scaler=None):
    """Save a checkpoint including the early-stopping state and all metrics of the epoch (to plot them later)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'seed': SEED,
        'epoch_metrics': epoch_metrics,
        'monitor_metric': EARLY_STOP_METRIC,
        'best_score': early_stopping.best_score,
        'es_counter': early_stopping.counter,
    }
    filepath = _prepare_save_path(filepath)
    torch.save(checkpoint, filepath)
    print(f" => Saved the resume checkpoint to: {filepath}")

def load_checkpoint(model, optimizer=None, scheduler=None, early_stopping=None, filepath=CHECKPOINT_FILE, resume_training=True, reset_training_state=False, scaler=None):
    """Load a checkpoint; when fine-tuning, optionally load only the weights and reset optimizer/scheduler/epoch."""
    start_epoch = 0
    if not resume_training:
        print("[*] RESUME_TRAINING = False. Training will start from epoch 1.")
        return start_epoch

    filepath = _resolve_checkpoint_path(filepath)

    if os.path.exists(filepath):
        print(f"[*] Found checkpoint '{filepath}'. Loading...")
        checkpoint = torch.load(filepath, map_location=device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        # strict=False: old checkpoints (e.g. a version without the contrastive loss) lack
        # image_proj/text_proj -> the matching weights still load and the new ones are re-initialised.
        load_result = _unwrap_model(model).load_state_dict(state_dict, strict=False)
        if load_result.missing_keys or load_result.unexpected_keys:
            print(f"[*] load_state_dict (strict=False): missing_keys={load_result.missing_keys}, "
                  f"unexpected_keys={load_result.unexpected_keys}")

        if reset_training_state:
            start_epoch = 0
            print("[*] Fine-tune mode: loading model weights only, resetting optimizer/scheduler/epoch.")
        else:
            if optimizer is not None and 'optimizer_state_dict' in checkpoint:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if scaler is not None and checkpoint.get('scaler_state_dict') is not None:
                scaler.load_state_dict(checkpoint['scaler_state_dict'])
            start_epoch = checkpoint.get('epoch', -1) + 1
            if early_stopping is not None:
                early_stopping.best_score = checkpoint.get('best_score', early_stopping.best_score)
                early_stopping.counter = checkpoint.get('es_counter', 0)

        print(f"[*] Restored successfully. Training will continue from epoch {start_epoch + 1}")
        if early_stopping is not None:
            print(f"[*] EarlyStopping state ({early_stopping.mode}): best_score = {early_stopping.best_score:.4f}, counter = {early_stopping.counter}")
    else:
        print(f"[*] Checkpoint '{filepath}' not found. Training will start from epoch 1.")
    return start_epoch

# ==========================================
# TRACK REAL CAPTION QUALITY (BLEU-4, pycocoevalcap) EACH EPOCH
# ==========================================
def build_val_reference_map(image_paths, captions, max_images=None):
    """Group the (standardised) captions of the same image into a list of reference strings, in the
    `gts` format that pycocoevalcap.bleu.Bleu.compute_score() expects (no manual tokenisation as in nltk)."""
    ref_map = {}
    order = []
    for path, cap in zip(image_paths, captions):
        if path not in ref_map:
            ref_map[path] = []
            order.append(path)
        ref_map[path].append(cap)
    if max_images is not None:
        order = order[:max_images]
    return {path: ref_map[path] for path in order}

@torch.no_grad()
def greedy_generate_tokens(model, image_tensor, max_length=SEQ_LENGTH):
    """Generate captions with greedy decoding (fast). ONLY used internally to track BLEU-4 each epoch
    during training. Not the final inference function - this notebook has no generate/
    evaluate captions (constrained decoding at inference is handled by the evaluation scripts)."""
    model_ = _unwrap_model(model)
    was_training = model_.training
    model_.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)
    memory = model_.encoder(image_tensor)
    generated = [START_IDX]

    for _ in range(max_length - 1):
        tgt = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(0)
        tgt_mask = model_.generate_square_subsequent_mask(tgt.size(1)).to(device)
        tgt_embed = model_.pos_encoder(model_.embedding(tgt))
        out = model_.transformer_decoder(tgt=tgt_embed, memory=memory, tgt_mask=tgt_mask)
        next_token = model_.fc_out(out)[0, -1, :].argmax(dim=-1).item()
        if next_token == END_IDX:
            break
        generated.append(next_token)

    if was_training:
        model_.train()
    return generated[1:]  # drop <start>

def evaluate_val_bleu4(model, vocab, ref_map, max_images=VAL_METRIC_MAX_SAMPLES, transforms=None):
    """Compute BLEU-4 with pycocoevalcap (the standard used for reporting in image-captioning papers/benchmarks
    captioning on COCO) on a subset of val images with greedy decode - does NOT use nltk because nltk
    BLEU was designed for machine translation evaluation, it is not the standard tool for captioning.
    Used to monitor / early-stop on real caption quality instead of only val_loss.
    Skip the PTBTokenizer step (needs Java) because captions are already normalised by `custom_standardization`
    already (lower-cased, punctuation removed) - the same way evaluate_baseline.py in the repo does it."""
    from pycocoevalcap.bleu.bleu import Bleu

    if transforms is None:
        transforms = val_transforms

    idx2word = {idx: word for word, idx in vocab.items()}
    gts, res = {}, {}

    for img_idx, image_path in enumerate(list(ref_map.keys())[:max_images]):
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            continue
        image_tensor = transforms(image)
        token_ids = greedy_generate_tokens(model, image_tensor)
        hyp_caption = " ".join(idx2word.get(tok, "<unk>") for tok in token_ids).strip()

        key = str(img_idx)
        gts[key] = ref_map[image_path]
        res[key] = [hyp_caption if hyp_caption else "<unk>"]

    if not res:
        return 0.0

    bleu_score, _ = Bleu(4).compute_score(gts, res)
    return bleu_score[3]  # BLEU-4

def train_model(train_loader, val_loader, vocab, object_token_ids=None, val_ref_map=None, id_to_obj_name=None):
    vocab_size = len(vocab)
    model = ImageCaptioningTransformer(
        vocab_size=vocab_size, embed_dim=EMBED_DIM,
        ff_dim=FF_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
        contrastive_proj_dim=CONTRASTIVE_PROJ_DIM
    ).to(device)

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"[*] Using {torch.cuda.device_count()} GPUs with nn.DataParallel")
        model = nn.DataParallel(model)

    # Label smoothing: reduces model overconfidence (one cause of hallucination).
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)

    num_training_steps = len(train_loader) * EPOCHS
    num_warmup_steps = int(0.05 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    print(f"[*] Mixed precision (AMP): {'ON' if amp_enabled else 'OFF'} (USE_AMP={USE_AMP}, cuda={torch.cuda.is_available()})")

    early_stop_mode = "min" if EARLY_STOP_METRIC == "loss" else "max"
    early_stopping = EarlyStoppingCallback(patience=5, save_path=BEST_MODEL_FILE, mode=early_stop_mode)

    # -------------------------
    # RESUME FROM CHECKPOINT (only to continue after the Kaggle kernel was interrupted, does not load the baseline)
    # -------------------------
    start_epoch = load_checkpoint(
        model,
        optimizer,
        scheduler,
        early_stopping,
        filepath=RESUME_CHECKPOINT_FILE,
        resume_training=RESUME_TRAINING,
        reset_training_state=FINE_TUNE_RESET_TRAINING_STATE,
        scaler=scaler
    )

    # Checkpoint without scheduler_state_dict (e.g. an old best model) -> the new scheduler starts at step 0 and
    # warm the LR up to its peak again, kicking an already converged model away. Fast-forward the scheduler to the number of steps already trained
    # (taken from the AdamW state), or build a new schedule with RESUME_PEAK_LR when the old one has finished.
    if start_epoch > 0 and RESUME_TRAINING and not FINE_TUNE_RESET_TRAINING_STATE:
        steps_per_epoch = len(train_loader)
        checkpoint_lr = optimizer.param_groups[0]['lr']
        adam_state = next(iter(optimizer.state.values()), {})
        done_steps = int(adam_state['step']) if 'step' in adam_state else start_epoch * steps_per_epoch
        if RESUME_PEAK_LR is not None:
            for group in optimizer.param_groups:
                group['lr'] = RESUME_PEAK_LR
                group['initial_lr'] = RESUME_PEAK_LR
            remaining_steps = steps_per_epoch * max(1, EPOCHS - start_epoch)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(0.05 * remaining_steps),
                num_training_steps=remaining_steps
            )
            print(f"[*] Resuming with a new LR schedule: peak {RESUME_PEAK_LR:g}, {remaining_steps} steps for the remaining {EPOCHS - start_epoch} epochs.")
        elif scheduler.last_epoch < done_steps:
            scheduler.last_epoch = done_steps - 1
            scheduler.step()
            print(f"[*] Checkpoint has no scheduler state -> fast-forwarding the scheduler to step {done_steps}.")
        resume_lr = optimizer.param_groups[0]['lr']
        print(f"[*] LR in the checkpoint: {checkpoint_lr:.2e} | LR on resume: {resume_lr:.2e}")
        if RESUME_PEAK_LR is None and checkpoint_lr < 0.01 * LEARNING_RATE and resume_lr > 0.1 * LEARNING_RATE:
            print("[!] WARNING: the checkpoint's LR schedule has finished (LR ~ 0) but the resume LR is large. "
                  "A sudden LR jump can make the loss rise or diverge; set RESUME_PEAK_LR (e.g. 2e-5).")

    log_interval = 200  # Large dataset, print less often to avoid flooding the screen

    for epoch in range(start_epoch, EPOCHS):
        # Seeding by epoch makes a resumed run reproduce the same batch/augmentation order.
        epoch_seed = SEED + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        print(f"\n--- Starting epoch {epoch+1:02d}/{EPOCHS} ---")
        _epoch_t0 = __import__("time").time()
        # USE_HALL_LOSS=False -> lambda=0, so hall_loss is still COMPUTED (for logging/monitoring via
        # object_token_ids shared with Dynamic Intercept/Contrastive) but NOT added to the loss.
        current_lambda_hall = get_hall_lambda(epoch, LAMBDA_HALL, HALL_LAMBDA_WARMUP_EPOCHS) if USE_HALL_LOSS else 0.0
        hall_min_step_index = int(HALL_LOSS_MIN_STEP_RATIO * SEQ_LENGTH)
        print(f"   [Hall Loss] lambda={current_lambda_hall:.5f} (max={LAMBDA_HALL}) | min_step_index={hall_min_step_index}")

        # -------------------------
        # PHASE 1: TRAINING
        # -------------------------
        current_lambda_con = get_contrastive_lambda(epoch, LAMBDA_CONTRASTIVE, CONTRASTIVE_WARMUP_EPOCHS) if USE_CONTRASTIVE_LOSS else 0.0
        if USE_CONTRASTIVE_LOSS:
            print(f"   [Contrastive] lambda={current_lambda_con:.5f} (max={LAMBDA_CONTRASTIVE}) | tau={CONTRASTIVE_TAU}")

        model.train()
        train_loss = 0
        train_ce_loss = 0
        train_hall_loss = 0
        train_con_loss = 0
        neg_valid_count, neg_total_count = 0, 0
        train_correct = 0
        train_tokens = 0

        for batch_idx, (images, captions, image_paths, gt_objects_batch, extra_pos_captions) in enumerate(train_loader):
            images, captions = images.to(device), captions.to(device)
            extra_pos_captions = extra_pos_captions.to(device)

            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full, neg_valid = build_negative_captions(
                        captions, gt_objects_batch, object_token_ids, id_to_obj_name,
                        num_negatives=NUM_NEGATIVES, num_structural=NUM_NEGATIVES_STRUCTURAL,
                        object_freq_weights=object_freq_weights, return_valid=True,
                    )
                    # Multi-positive contrastive: append (K_POS_PER_IMAGE - 1) REAL captions of the SAME image (extra
                    # positives, used only for contrastive learning) from the synthetic negatives, decode BOTH
                    # in the SAME model() call, reusing the existing decode-N-captions mechanism
                    # (DataParallel compatible), no change to forward() needed.
                    extra_pos_decoder_input = extra_pos_captions[:, :, :-1]
                    neg_decoder_input = neg_captions_full[:, :, :-1]
                    combined_decoder_input = torch.cat([extra_pos_decoder_input, neg_decoder_input], dim=1)
                    # Call through `model` (nn.DataParallel if >1 GPU), NOT through the unwrapped model_ -
                    # forward() now encodes the image once, then decodes both positives and negatives IN THE SAME
                    # a single call, so DataParallel scatter/gather works correctly over this whole branch.
                    outputs, pos_hidden, image_embed, pos_text_embed, combined_text_embed = model(
                        images, decoder_input, neg_captions=combined_decoder_input, return_hidden=True
                    )
                else:
                    outputs = model(images, decoder_input)
                # Intercept in the decoder AT TRAINING TIME: lower the logits of tokens of absent objects
                # before computing CE/hall_loss, so the model learns directly from the blocking instead of only being
                # penalised indirectly through hall_loss.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                        outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                    )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    # Multi-positive contrastive (SupCon - Khosla et al. 2020): split extra positives and
                    # the negatives from combined_text_embed, concatenated with pos_text_embed (the CE anchor) into
                    # ALL real positives of this image, then compute the multi-positive InfoNCE; captions
                    # REAL captions of other images in the batch automatically act as negatives (in-batch, free).
                    num_extra_pos = extra_pos_decoder_input.size(1)
                    extra_pos_text_embed = combined_text_embed[:, :num_extra_pos, :]
                    synth_neg_text_embed = combined_text_embed[:, num_extra_pos:, :]
                    all_pos_text_embed = torch.cat([pos_text_embed.unsqueeze(1), extra_pos_text_embed], dim=1)
                    all_text_embed = torch.cat([all_pos_text_embed, synth_neg_text_embed], dim=1)
                    con_loss = multi_positive_infonce_loss(
                        image_embed, all_text_embed, num_positives=1 + num_extra_pos, tau=CONTRASTIVE_TAU,
                        negative_valid=neg_valid,
                    )
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)

            predictions = outputs.argmax(dim=-1)
            mask = (targets != PAD_IDX)
            correct = (predictions == targets) & mask

            train_correct += correct.sum().item()
            train_tokens += mask.sum().item()

            scaler.scale(loss).backward()
            # unscale before clipping: clipping must use the real gradients, not the AMP-scaled ones.
            scaler.unscale_(optimizer)
            # Gradient clipping: stabilises gradients when training the decoder from scratch (no pretrained baseline).
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            train_ce_loss += ce_loss.item()
            train_hall_loss += hall_loss.item()
            train_con_loss += con_loss.item()
            if USE_CONTRASTIVE_LOSS:
                neg_valid_count += int(neg_valid.sum().item())
                neg_total_count += neg_valid.numel()

            if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
                batch_acc = (correct.sum().item() / mask.sum().item()) * 100 if mask.sum().item() > 0 else 0.0
                print(f"   [Epoch {epoch+1:02d}] Batch {batch_idx+1:05d}/{len(train_loader)} | Loss: {loss.item():.4f} | CE: {ce_loss.item():.4f} | Hall: {hall_loss.item():.6f} | Con: {con_loss.item():.6f} | Acc: {batch_acc:.2f}%")

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_ce_loss = train_ce_loss / len(train_loader)
        epoch_train_hall_loss = train_hall_loss / len(train_loader)
        epoch_train_con_loss = train_con_loss / len(train_loader)
        epoch_train_acc = (train_correct / train_tokens) * 100

        # -------------------------
        # PHASE 2: VALIDATION
        # -------------------------
        model.eval()
        val_loss = 0
        val_ce_loss = 0
        val_hall_loss = 0
        val_con_loss = 0
        val_correct = 0
        val_tokens = 0

        if val_loader.generator is not None:
            val_loader.generator.manual_seed(SEED + 100_000 + epoch)

        with torch.no_grad():
            for images, captions, image_paths, gt_objects_batch, extra_pos_captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                extra_pos_captions = extra_pos_captions.to(device)

                decoder_input = captions[:, :-1]
                targets = captions[:, 1:]

                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full, neg_valid = build_negative_captions(
                        captions, gt_objects_batch, object_token_ids, id_to_obj_name,
                        num_negatives=NUM_NEGATIVES, num_structural=NUM_NEGATIVES_STRUCTURAL,
                        object_freq_weights=object_freq_weights, return_valid=True,
                    )
                    extra_pos_decoder_input = extra_pos_captions[:, :, :-1]
                    neg_decoder_input = neg_captions_full[:, :, :-1]
                    combined_decoder_input = torch.cat([extra_pos_decoder_input, neg_decoder_input], dim=1)
                    outputs, pos_hidden, image_embed, pos_text_embed, combined_text_embed = model(
                        images, decoder_input, neg_captions=combined_decoder_input, return_hidden=True
                    )
                else:
                    outputs = model(images, decoder_input)
                # Consistent with training: val_loss/val_acc reflect the "blocked"
                # distribution the model actually uses when generating.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                    outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    num_extra_pos = extra_pos_decoder_input.size(1)
                    extra_pos_text_embed = combined_text_embed[:, :num_extra_pos, :]
                    synth_neg_text_embed = combined_text_embed[:, num_extra_pos:, :]
                    all_pos_text_embed = torch.cat([pos_text_embed.unsqueeze(1), extra_pos_text_embed], dim=1)
                    all_text_embed = torch.cat([all_pos_text_embed, synth_neg_text_embed], dim=1)
                    con_loss = multi_positive_infonce_loss(
                        image_embed, all_text_embed, num_positives=1 + num_extra_pos, tau=CONTRASTIVE_TAU,
                        negative_valid=neg_valid,
                    )
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)
                val_loss += loss.item()
                val_ce_loss += ce_loss.item()
                val_hall_loss += hall_loss.item()
                val_con_loss += con_loss.item()

                predictions = outputs.argmax(dim=-1)
                mask = (targets != PAD_IDX)
                val_correct += ((predictions == targets) & mask).sum().item()
                val_tokens += mask.sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_ce_loss = val_ce_loss / len(val_loader)
        epoch_val_hall_loss = val_hall_loss / len(val_loader)
        epoch_val_con_loss = val_con_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_tokens) * 100

        # Real BLEU-4 on a val subset (greedy decode), computed only when needed for monitoring/early stopping.
        epoch_val_bleu4 = None
        if EARLY_STOP_METRIC == "bleu4" and val_ref_map:
            epoch_val_bleu4 = evaluate_val_bleu4(model, vocab, val_ref_map, max_images=VAL_METRIC_MAX_SAMPLES)

        monitor_score = epoch_val_bleu4 if EARLY_STOP_METRIC == "bleu4" else epoch_val_loss

        # -------------------------
        # REPORT AND SAVE CHECKPOINT
        # -------------------------
        print(f"\n=> EPOCH {epoch+1:02d} SUMMARY:")
        print(f"   Train Loss: {epoch_train_loss:.4f} | CE: {epoch_train_ce_loss:.4f} | Hall: {epoch_train_hall_loss:.6f} | Con: {epoch_train_con_loss:.6f} | Train Acc: {epoch_train_acc:.2f}%")
        if USE_CONTRASTIVE_LOSS and neg_total_count:
            print(f"   Usable negatives (different from the positive): {100 * neg_valid_count / neg_total_count:.1f}% ({neg_valid_count}/{neg_total_count} slots)")
        bleu_str = f" | BLEU-4: {epoch_val_bleu4:.4f}" if epoch_val_bleu4 is not None else ""
        print(f"   Val Loss  : {epoch_val_loss:.4f} | CE: {epoch_val_ce_loss:.4f} | Hall: {epoch_val_hall_loss:.6f} | Con: {epoch_val_con_loss:.6f} | Val Acc  : {epoch_val_acc:.2f}%{bleu_str}")

        # 1. Save the best model by EARLY_STOP_METRIC (BLEU-4 by default, not val_loss)
        early_stopping(monitor_score, model, optimizer, epoch, scheduler, scaler)

        # 2. Always save the last checkpoint after each epoch so training can resume if the Kaggle kernel dies
        epoch_metrics = {
            "val_loss": epoch_val_loss,
            "val_ce_loss": epoch_val_ce_loss,
            "val_hall_loss": epoch_val_hall_loss,
            "val_con_loss": epoch_val_con_loss,
            "val_acc": epoch_val_acc,
            "val_bleu4": epoch_val_bleu4,
        }
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, CHECKPOINT_FILE, scaler=scaler)

        # Stop safely before the Kaggle session limit (TRAIN_TIME_BUDGET_HOURS); this epoch's checkpoint is already saved above.
        if TRAIN_TIME_BUDGET_HOURS and epoch + 1 < EPOCHS:
            _now = __import__("time").time()
            _epoch_h = (_now - _epoch_t0) / 3600
            _elapsed_h = (_now - NOTEBOOK_START_TIME) / 3600
            if _elapsed_h + 1.1 * _epoch_h > TRAIN_TIME_BUDGET_HOURS:
                print(f"[Time] Ran {_elapsed_h:.2f} hours, last epoch took {_epoch_h * 60:.0f} minutes: not enough for "
                      f"epoch {epoch + 2} within {TRAIN_TIME_BUDGET_HOURS} hours. Stopping after epoch {epoch + 1}; next session set "
                      f"RESUME_TRAINING=True, RESUME_CHECKPOINT_FILE=<{os.path.basename(CHECKPOINT_FILE)} in the output>.")
                break

        if early_stopping.early_stop:
            print(f"⚠️ Early stopping at epoch {epoch+1}: {EARLY_STOP_METRIC} did not improve for several epochs.")
            break

    return model, optimizer

In [ ]:
# ==========================================
# DIAGNOSTIC: DO KARPATHY FILE NAMES MATCH THE COCO INSTANCES?
# ==========================================
def diagnose_hall_label_coverage(image_paths, image_objects, label="train"):
    """Early warning for the MOST COMMON reason why hallucination_object_loss/
    apply_decoder_object_intercept to have no effect: HallDebug prints `gt_images=0 |
    present_images=0` in EVERY batch. On Flickr30k the labels come from the captions, so the match should be close to 100%; a low value means caption parsing failed,
    or the file names in the instances file do not match the real image file names (some Kaggle datasets rename
    file name, without the COCO_train2014_/COCO_val2014_ prefix)."""
    if not image_paths:
        return
    sample_basenames = [os.path.basename(p) for p in image_paths[:2000]]
    matched = sum(1 for name in sample_basenames if name in image_objects)
    coverage = matched / len(sample_basenames) * 100
    print(f"   [HallCheck-{label}] Karpathy <-> COCO instances file-name match: {matched}/{len(sample_basenames)} ({coverage:.1f}%) on the first {len(sample_basenames)} images.")
    if coverage < 5:
        example_img = sample_basenames[0]
        example_ann = next(iter(image_objects.keys()), None)
        print(f"   [!] WARNING: almost no file names match -> hallucination_object_loss AND apply_decoder_object_intercept will always be 0 (no effect); the model is training as a plain baseline.")
        print(f"       Example Karpathy image name: '{example_img}'")
        print(f"       Example key in COCO instances: '{example_ann}'")
        print(f"       -> Check how captions are read (load_flickr_data) and build_caption_objects.")

# ==========================================
# EXECUTION (MAIN) - TRAIN FROM SCRATCH
# ==========================================
if __name__ == "__main__":
    print("0. Preparing the dataset and vocabulary for training from scratch on Flickr30k...")
    print(f"   SEED={SEED} | cuDNN deterministic=True | benchmark=False")
    print(f"   FF_DIM={FF_DIM} | NUM_LAYERS={NUM_LAYERS} | LR={LEARNING_RATE} | EPOCHS={EPOCHS} | LABEL_SMOOTHING={LABEL_SMOOTHING}")
    print(f"   EARLY_STOP_METRIC={EARLY_STOP_METRIC} (VAL_METRIC_MAX_SAMPLES={VAL_METRIC_MAX_SAMPLES})")
    print(f"   TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | MODE={TRAIN_INTERCEPT_MODE} | MAX_PENALTY={TRAIN_INTERCEPT_MAX_PENALTY} | WARMUP={TRAIN_INTERCEPT_WARMUP_EPOCHS}")
    print(f"   USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | LAMBDA_CONTRASTIVE={LAMBDA_CONTRASTIVE} | TAU={CONTRASTIVE_TAU} | WARMUP={CONTRASTIVE_WARMUP_EPOCHS} | POOLING_MODE={POOLING_MODE}")

    # 0.1 Load the Train images and captions of the Karpathy split
    train_imgs, train_caps = load_flickr_data(TRAIN_SPLITS)
    print(f"   Split source: {SPLIT_SOURCE} | image folder: {FLICKR_IMG_DIR}")

    # 0.2 Load the Val images and captions of the Karpathy split
    val_imgs, val_caps = load_flickr_data("val")

    # 0.3 Build (or load) the vocabulary from the Train split and save it to file (so it can be reloaded)
    vocab = build_or_load_vocab(train_caps, min_freq=MIN_WORD_FREQ, vocab_file=VOCAB_FILE)
    vocab_size = len(vocab)

    # USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS:
    # build the object/COCO-instances setup when ANY component needs it, not only HalluLoss
    # (Dynamic Decoder Intercept and the contrastive loss share this data).
    object_token_ids = build_object_token_ids(vocab) if USE_OBJECT_GROUNDING else {}
    val_image_objects = build_caption_objects(val_imgs, val_caps) if (USE_OBJECT_GROUNDING and USE_CAPTION_OBJECT_LABELS) else {}
    train_image_objects = build_caption_objects(train_imgs, train_caps) if (USE_OBJECT_GROUNDING and USE_CAPTION_OBJECT_LABELS) else {}
    if USE_OBJECT_GROUNDING:
        print(f"   Object grounding: USE_HALL_LOSS={USE_HALL_LOSS} (lambda={LAMBDA_HALL}) | "
              f"TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | "
              f"object labels taken from the 5 captions per image (no YOLO)={USE_CAPTION_OBJECT_LABELS}")
        if USE_CAPTION_OBJECT_LABELS:
            diagnose_hall_label_coverage(train_imgs, train_image_objects, label="train")
            diagnose_hall_label_coverage(val_imgs, val_image_objects, label="val")

    # The contrastive loss reuses exactly
    # object_token_ids built above (via USE_OBJECT_GROUNDING) - needs a non-empty object_token_ids.
    id_to_obj_name = build_object_id_to_name(object_token_ids) if USE_CONTRASTIVE_LOSS else {}
    # REAL category frequencies in the ground-truth captions (train_caps), used as weights
    # to pick hallucination objects, instead of sampling uniformly over the 80 categories.
    # (based on Rohrbach et al. 2018).
    object_freq_weights = build_object_frequency_weights(train_caps, object_token_ids) if USE_CONTRASTIVE_LOSS else {}
    if USE_CONTRASTIVE_LOSS:
        print(f"   Contrastive loss: lambda={LAMBDA_CONTRASTIVE}, tau={CONTRASTIVE_TAU}, "
              f"warmup_epochs={CONTRASTIVE_WARMUP_EPOCHS}, proj_dim={CONTRASTIVE_PROJ_DIM}, " f"pooling_mode={POOLING_MODE}")
        if not object_token_ids:
            print("   [!] WARNING: object_token_ids is empty -> build_negative_captions will have no "
                  "object to build negatives from, so the contrastive loss will always be ~0 (no effect).")

    # 0.4 Check pycocoevalcap before training (fail fast); if missing, fall back to "loss"
    # instead of crashing in the middle of the first epoch when evaluate_val_bleu4 is called.
    if EARLY_STOP_METRIC == "bleu4":
        try:
            from pycocoevalcap.bleu.bleu import Bleu  # noqa: F401
        except ImportError:
            print("[!] pycocoevalcap not found -> falling back to EARLY_STOP_METRIC='loss'. "
                  "Install `pip install pycocoevalcap` to use the standard COCO-caption BLEU-4 for early stopping.")
            EARLY_STOP_METRIC = "loss"

    # 0.5 Reference map (several captions per image) to compute real BLEU-4 each epoch, if enabled.
    val_ref_map = None
    if EARLY_STOP_METRIC == "bleu4":
        val_ref_map = build_val_reference_map(val_imgs, val_caps, max_images=VAL_METRIC_MAX_SAMPLES)
        print(f"   -> BLEU-4 will be computed each epoch on {len(val_ref_map)} val images (greedy decode).")

    # 1. Create the Dataset & DataLoader
    _k_pos = K_POS_PER_IMAGE if USE_CONTRASTIVE_LOSS else 1  # multi-positive: only take extra
    # positives only when the contrastive loss is actually used, to avoid useless tokenisation when it is off.
    train_dataset = FlickrDataset(train_imgs, train_caps, vocab, transforms=train_transforms, image_objects=train_image_objects, k_pos_per_image=_k_pos)
    val_dataset = FlickrDataset(val_imgs, val_caps, vocab, transforms=val_transforms, image_objects=val_image_objects, k_pos_per_image=_k_pos)

    print("1. Creating DataLoaders...")
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)
    val_generator = torch.Generator()
    val_generator.manual_seed(SEED + 1)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=val_generator
    )

    print("2. Training CLIP-ViT-B/16 + Transformer FROM SCRATCH with the hallucination-aware loss on Flickr30k...")
    model, optimizer = train_model(
        train_loader,
        val_loader,
        vocab=vocab,
        object_token_ids=object_token_ids,
        val_ref_map=val_ref_map,
        id_to_obj_name=id_to_obj_name,
    )

    print("3. Training finished all requested epochs.")